<a href="https://colab.research.google.com/github/bfumia/beginning-bioinformatics/blob/SWISNF/SWI_SNF_Chromatin_Conservation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!apt-get install muscle
!pip install biopython pandas matplotlib

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  muscle
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 244 kB of archives.
After this operation, 709 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 muscle amd64 1:3.8.1551-2build1 [244 kB]
Fetched 244 kB in 0s (2,778 kB/s)
Selecting previously unselected package muscle.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../muscle_1%3a3.8.1551-2build1_amd64.deb ...
Unpacking muscle (1:3.8.1551-2build1) ...
Setting up muscle (1:3.8.1551-2build1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.0 MB/s eta 0:00:00


In [33]:
import pandas as pd

swi_snf = pd.DataFrame([
    ["BRM",    "AT2G46020", "Q9FYC2"],
    ["CHR12",  "AT3G06050", "Q9SDR4"],
    ["CHR23",  "AT5G19310", "Q9FXE9"],
    ["SWI3A",  "AT2G33610", "Q9LU60"],
    ["SWI3B",  "AT2G47620", "Q9LMI1"],
    ["SWI3C",  "AT1G21700", "Q9S9L0"],
    ["SWI3D",  "AT1G77670", "Q9SPL2"],
    ["ARP7",   "AT3G60830", "Q9LPH5"],
    ["SWP73A", "AT5G14170", "Q9FKG9"],
    ["SWP73B", "AT5G19380", "Q9FKG8"],
], columns=["SWI/SNF Subunit", "TAIR_ID", "UniProt"])

swi_snf

,SWI/SNF Subunit,TAIR_ID,UniProt
0,BRM,AT2G46020,Q9FYC2
1,CHR12,AT3G06050,Q9SDR4
2,CHR23,AT5G19310,Q9FXE9
3,SWI3A,AT2G33610,Q9LU60
4,SWI3B,AT2G47620,Q9LMI1
5,SWI3C,AT1G21700,Q9S9L0
6,SWI3D,AT1G77670,Q9SPL2
7,ARP7,AT3G60830,Q9LPH5
8,SWP73A,AT5G14170,Q9FKG9
9,SWP73B,AT5G19380,Q9FKG8


In [34]:
import requests

def uniprot_fasta(uid, file):
  url = f"https://rest.uniprot.org/uniprotkb/{uid}.fasta"
  r = requests.get(url)
  if r.status_code == 200:
    open(file, "wb").write(r.content)
    print("downloaded", file)
  else:
    print("failed", uid)
for _, row in swi_snf.iterrows():
  uniprot_fasta(row["UniProt"], f"{row['SWI/SNF Subunit']}.fasta")

downloaded BRM.fasta
downloaded CHR12.fasta
downloaded CHR23.fasta
downloaded SWI3A.fasta
downloaded SWI3B.fasta
downloaded SWI3C.fasta
downloaded SWI3D.fasta
downloaded ARP7.fasta
downloaded SWP73A.fasta
downloaded SWP73B.fasta


In [35]:
from Bio.Blast import NCBIWWW
import os

def run_blast(query_file, out_xml):
    seq = open(query_file).read()
    result = NCBIWWW.qblast(
        "blastp",
        "nr",
        seq,
        entrez_query="Oryza sativa[organism]"
    )
    open(out_xml, "w").write(result.read())
    print("BLAST complete:", out_xml)

for _, row in swi_snf.iterrows():
    name = row["SWI/SNF Subunit"]
    fasta_file = f"{name}.fasta"
    blast_xml_file = f"{name}_blast.xml"

    if not os.path.exists(blast_xml_file):
        run_blast(fasta_file, blast_xml_file)
    else:
        print(f"BLAST results already exist for {name}: {blast_xml_file}")


BLAST results already exist for BRM: BRM_blast.xml
BLAST results already exist for CHR12: CHR12_blast.xml
BLAST results already exist for CHR23: CHR23_blast.xml
BLAST results already exist for SWI3A: SWI3A_blast.xml
BLAST results already exist for SWI3B: SWI3B_blast.xml
BLAST results already exist for SWI3C: SWI3C_blast.xml
BLAST results already exist for SWI3D: SWI3D_blast.xml
BLAST results already exist for ARP7: ARP7_blast.xml
BLAST results already exist for SWP73A: SWP73A_blast.xml
BLAST results already exist for SWP73B: SWP73B_blast.xml


In [38]:
from Bio.Blast import NCBIXML
import pandas as pd

top_hits = {}

def parse_top_hits(xml_file, n=3):
    record = NCBIXML.read(open(xml_file))
    hits = []
    for aln in record.alignments[:n]:
        hits.append([aln.hit_def, aln.accession])
    return pd.DataFrame(hits, columns=["Description", "Accession"])

for _, row in swi_snf.iterrows():
    name = row["SWI/SNF Subunit"]
    df = parse_top_hits(f"{name}_blast.xml")
    top_hits[name] = df
    print("=========", name, "=========")
    display(df)

========= BRM =========


,Description,Accession
0,"pheophorbide a oxygenase, chloroplastic-like [...",XP_015633218
1,hypothetical protein OsI_10012 [Oryza sativa I...,EAY88539
2,"RecName: Full=Pheophorbide a oxygenase, chloro...",Q0DV66


========= CHR12 =========


,Description,Accession
0,hypothetical protein OsI_27074 [Oryza sativa I...,EAZ04892
1,dirigent protein 5-like [Oryza sativa Japonica...,XP_025882837
2,dirigent protein 5-like [Oryza sativa Japonica...,XP_066168127


========= CHR23 =========


...

,Description,Accession
0,pentatricopeptide repeat-containing protein At...,NP_001409824
1,hypothetical protein OsJ_25936 [Oryza sativa J...,EAZ41414
2,hypothetical protein OsI_27705 [Oryza sativa I...,EAZ05488


========= SWP73A =========


,Description,Accession
0,hypothetical protein OsI_00782 [Oryza sativa I...,EEC70119
1,autophagy-related protein 3 [Oryza sativa Japo...,XP_015613687
2,LOW QUALITY PROTEIN: autophagy-related protein...,XP_015613096


========= SWP73B =========


,Description,Accession
0,"hypothetical protein EE612_014734, partial [Or...",KAB8089744
1,uncharacterized protein [Oryza sativa Japonica...,XP_015630622
2,"oxidoreductase, zinc-binding dehydrogenase fam...",ABF93481
